# Clean NVE avalanche events

`01_ingestion/nve_avalanche_events.py` writes to `02_data/raw/` like any other
source, but `backend/transform.py` deliberately excludes every `nve_*` source
from the `records` table (it's ~30x the row count of everything else, and
`records` feeds a table that gets uploaded to a live bucket — see the comment
in `build_tables()`, and `05_tests/test_records_excludes_nve.py`).

So this notebook reads the raw file directly instead of going through
`storage.load("records")`, shapes it the same way the other two cleaning
notebooks shape their data, and saves it under its own name: `storage.load
("nve_avalanche_events_clean")` — a page can read it, `records` never sees it.

In [ ]:
import sys

sys.path.insert(0, "..")  # the notebook runs from 03_notebooks/

import pandas as pd

from backend import config, storage

storage.tables()

Read the raw file directly — not `storage.load("records")`, since NVE data
is never in there. `storage.read_file()` is the same reader `transform.py`
uses internally; matched against `storage.DATA_SUFFIXES` so it finds the file
regardless of which `FILE_FORMAT` wrote it.

In [ ]:
raw_path = next(
    (p for p in config.RAW_DIR.glob("nve_avalanche_events.*") if p.suffix.lower() in storage.DATA_SUFFIXES),
    None,
)
raw = storage.read_file(raw_path) if raw_path else pd.DataFrame()
raw.head()

Clean `raw` into `nve_avalanche_events_clean`:

1. Lower-case every column name.
2. Cast `døde` to `int64` — no value recorded means zero, not unknown (most
   NVE events have no casualties at all; this is a hazard registry, not an
   accident report).
3. Parse `dato` to a real `datetime64` dtype, then derive `year`, `month`
   (Norwegian month name) and `day` (Norwegian weekday name) from it. `year`
   uses pandas' nullable `Int64` (capital I) rather than plain `int64` — some
   rows have no date at all, and a plain int column can't hold both a missing
   value and stay an integer type; a bare `int64` would silently become
   `float64` (`2001.0` instead of `2001`) the moment any row is missing.
4. Drop rows above 74°N — same reasoning as the other two cleaning
   notebooks: the map page's Kartverket tiles only cover mainland Norway, not
   Svalbard (this dataset has no `område`/`fylke` column to filter on by
   name like the other two, so this uses latitude directly instead).

In [ ]:
NORWEGIAN_MONTHS = {
    1: "Januar", 2: "Februar", 3: "Mars", 4: "April", 5: "Mai", 6: "Juni",
    7: "Juli", 8: "August", 9: "September", 10: "Oktober", 11: "November", 12: "Desember",
}
NORWEGIAN_WEEKDAYS = {
    0: "Mandag", 1: "Tirsdag", 2: "Onsdag", 3: "Torsdag", 4: "Fredag", 5: "Lørdag", 6: "Søndag",
}

clean = pd.DataFrame()

if raw.empty:
    print("No `nve_avalanche_events` raw file yet — run `make ingestion` first.")
else:
    clean = raw.copy()
    clean.columns = clean.columns.str.lower()

    clean["døde"] = clean["døde"].fillna(0).astype("int64")

    clean["dato"] = pd.to_datetime(clean["dato"], format="%Y-%m-%d")
    clean["year"] = clean["dato"].dt.year.astype("Int64")
    clean["month"] = clean["dato"].dt.month.map(NORWEGIAN_MONTHS)
    clean["day"] = clean["dato"].dt.weekday.map(NORWEGIAN_WEEKDAYS)

    clean = clean[clean["latitude"] < 74]

    storage.save("nve_avalanche_events_clean", clean)

clean.head()